# Vineyard Canopy Segmentation Training on Google Colab (YOLO26-Seg)

Train the flagship **YOLO26 Large Segmentation model (`yolo26l-seg.pt`)** on Google Colab with GPU acceleration (T4 / A100 / L4).

### Model & Dataset:
- **Architecture:** Ultralytics YOLO26 Large Instance Segmentation (`yolo26l-seg`)
- **Dataset:** EU ICAERUS Vine-Specific Dataset (59 train / 26 val aerial crops, 1,093 vine polygons)
- **Task:** Crisp polygon mask extraction for aerial vineyard canopy mapping.

## 1. Check GPU Environment

In [ ]:
!nvidia-smi
!pip install --upgrade pip ultralytics opencv-python shapely

## 2. Upload and Unzip Dataset
Upload `dataset_yolo.zip` (generated locally via `python scripts/prepare_colab_datasets.py`, only ~28 MB).

It contains:
```
dataset/
  ├── data.yaml
  ├── images/ (train: 59, val: 26)
  └── labels/ (train: 59, val: 26)
```

In [ ]:
# Unzip the uploaded dataset
!unzip -q dataset_yolo.zip -d /content/dataset
!ls -la /content/dataset

## 3. Verify Dataset Configuration YAML

In [ ]:
# Verify data.yaml
!cat /content/dataset/data.yaml

## 4. Train YOLO Segmentation Model
Optimized for aerial UAV imagery with rotation, flip, and mosaic augmentations.

In [ ]:
from ultralytics import YOLO

# Select model variant: "yolo26n-seg.pt" (Nano) or "yolo26l-seg.pt" (Large)
# Docs: https://docs.ultralytics.com/tasks/segment
MODEL_NAME = "yolo26n-seg.pt"
model = YOLO(MODEL_NAME)

# Train with aerial-specific augmentations
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=512,          # Native dataset resolution (~502x502)
    batch=8,            # Fits comfortably on GPU
    device=0,
    degrees=180,        # 360-degree rotation invariance for UAV imagery
    fliplr=0.5,
    flipud=0.5,
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=10,
    workers=4,
    project="vineyard_runs",
    name="yolo26_canopy_colab",
    save=True,
)


## 5. Download the Trained Weights
The best weights will be saved at `vineyard_runs/yolo26l_canopy_colab/weights/best.pt`.
Download it and place it into `weights/best.pt` in the local repository.

In [ ]:
from google.colab import files
import os, glob

# Download the trained best checkpoint
ckpts = glob.glob("vineyard_runs/**/weights/best.pt", recursive=True)
if ckpts:
    best_weights = ckpts[-1]
    print(f"Downloading {best_weights} ({os.path.getsize(best_weights) / 1024 / 1024:.1f} MB)...")
    files.download(best_weights)
else:
    print("Weights file not found. Check vineyard_runs directory.")
